# 🩺 Triavia — Exploratory Data Analysis

Sections: overview · class distribution · vitals by level · correlation · feature engineering preview

> Run from the `notebooks/` folder (or the project root). Needs `pip install -r requirements-dev.txt`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px
from model.training.data_generator import generate_dataset
from model.train import engineer_features
from model.training.config import NUMERIC_FEATURES, TRIAGE_LABELS
sns.set_theme(style='whitegrid'); pd.set_option('display.max_columns', None)
df = generate_dataset(n=5000)
print(df.shape); df.head()

## 1. Dataset Overview

In [ ]:
print(df.dtypes)
print('\nNull counts (≈3% missing values are injected on purpose — the pipeline imputes them):')
print(df.isnull().sum()[df.isnull().sum()>0])
df[NUMERIC_FEATURES].describe().round(2)

## 2. Class Distribution

In [ ]:
counts = df['triage_level'].value_counts().sort_index()
labels = [TRIAGE_LABELS[i] for i in counts.index]
colors = ['#22c55e','#f59e0b','#ef4444','#7c3aed']
fig = px.bar(x=labels, y=counts.values, color=labels, color_discrete_sequence=colors,
             title='Triage Level Distribution', labels={'x':'Level','y':'Count'})
fig.update_layout(showlegend=False); fig.show()
print((counts/len(df)*100).round(1).astype(str)+'%')

## 3. Vital Signs by Triage Level

In [ ]:
df['label'] = df['triage_level'].map(TRIAGE_LABELS)
vitals = ['heart_rate','systolic_bp','oxygen_saturation','temperature','respiratory_rate','pain_scale']
palette = dict(zip([TRIAGE_LABELS[i] for i in range(4)], colors))
fig, axes = plt.subplots(2, 3, figsize=(16,10))
for ax, v in zip(axes.flatten(), vitals):
    sns.boxplot(data=df, x='label', y=v, hue='label', palette=palette, legend=False, ax=ax, width=.55,
                order=[TRIAGE_LABELS[i] for i in range(4)])
    ax.set_title(v.replace('_',' ').title(), fontweight='bold'); ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)
plt.suptitle('Vital Signs by Triage Level', fontsize=14, fontweight='bold'); plt.tight_layout(); plt.show()

## 4. Correlation Heatmap

In [ ]:
corr = df[NUMERIC_FEATURES+['triage_level']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(11,9))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=.5, ax=ax, square=True)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold'); plt.tight_layout(); plt.show()
print('\nCorrelation with triage_level:')
print(corr['triage_level'].drop('triage_level').sort_values(ascending=False))

## 5. Engineered Features

In [ ]:
df_e = engineer_features(df)
palette_map = {TRIAGE_LABELS[i]: colors[i] for i in range(4)}
fig = px.scatter(df_e, x='shock_index', y='oxygen_saturation', color='label',
                 color_discrete_map=palette_map, opacity=.55,
                 title='Shock Index vs SpO₂ by Triage Level',
                 labels={'shock_index':'Shock Index (HR/SBP)','oxygen_saturation':'SpO₂ (%)'})
fig.update_traces(marker=dict(size=4)); fig.show()
print(df_e.groupby('label')['shock_index'].describe().round(3))